# ATM 407: anatomy of an atmospheric column model

This lab treats the SCM as the physics column that would sit inside a dynamical core. You will examine the vertical coordinate, diagnose static stability, separate resolved forcing from parameterized tendencies, and use the pressure-coordinate thermodynamic equation to study how large-scale ascent competes with convective adjustment.

The SCM has no horizontal pressure-gradient force, advection, Coriolis acceleration, or resolved vertical motion. Keep that limitation in mind when connecting these results to atmospheric dynamics.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanwellmeyer/GCM/blob/main/notebooks/02_experiments_atm407.ipynb)

## Colab setup

Run this cell first. It installs the current model and verifies that Python can import it.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

incolab = 'google.colab' in sys.modules
if incolab:
    root = Path('/content/GCM')
    if not root.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/evanwellmeyer/GCM.git', str(root),
        ], check=True)
else:
    root = Path.cwd().resolve()
    while root != root.parent and not (root / 'pyproject.toml').exists():
        root = root.parent

if not (root / 'pyproject.toml').exists():
    raise FileNotFoundError('Open this notebook from inside the GCM repository.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from importlib import metadata, util

try:
    metadata.version('gcm-scm')
    installed = util.find_spec('matplotlib') is not None
except metadata.PackageNotFoundError:
    installed = False

if not installed:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--quiet', '-e', f'{root}[plot]',
    ], check=True)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import scm
print('SCM ready from', Path(scm.__file__).resolve())

In [ ]:
from copy import deepcopy
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

from scm.column_model import initial_state, physics_step, run, update_derived
from scm.configuration import extract_param_overrides, load_run_config
from scm.ensemble import default_params
from scm.thermo import Rd, cp, g, geopotential, make_grid, relative_humidity

torch.manual_seed(0)
device = torch.device('cpu')
print('device:', device)

## Model controls

The notebook loads the accepted `mf_response_v3` 20-level reference state generated with multiband radiation and mass-flux convection. Its late-window TOA imbalance is +0.56 W m$^{-2}$, surface imbalance is +0.31 W m$^{-2}$, and 50-day temperature drift is 0.006 K. CAPE is about 1,434 J kg$^{-1}$, only 3.5% of atmospheric mass is at or above 95% relative humidity, deep convection produces most of the precipitation, and no mass-flux or tendency caps are active. These checks support using it as a balanced teaching reference, although it remains an idealized radiative-convective column rather than an observed tropical sounding. When another resolution is requested, temperature and water fields are interpolated in sigma coordinates and column water is conserved; the result is an initial guess, not an equilibrium on the new grid.


In [ ]:
experiment = {
    'nlevels': 20,
    'dt': 900.0,
    'days': 3,
    'diagnostic_hours': 3,
    'radiation_steps': 8,
    'surface_temperature': 290.0,
    'surface_pressure': 100000.0,
    'solar_constant': 1360.0,
    'zenith_factor': 0.25,
    'ocean_depth': 50.0,
    'surface_albedo': 0.32,
    'wind_speed': 5.0,
}

referencemetadata = json.loads(
    (root / 'notebooks/data/atm407_equilibrium_20level.json').read_text()
)
print('reference configuration:', referencemetadata['configuration_label'])
print(f"reference surface temperature: {referencemetadata['surface_temperature_k']:.2f} K")
print(f"reference CAPE: {referencemetadata['cape_jkg']:.0f} J kg-1")
print(f"reference precipitation: {referencemetadata['precipitation_mmday']:.2f} mm day-1")
print(f"mass at or above 95% RH: {referencemetadata['rh95_mass_fraction']:.0%}")
print(f"mass-flux cap-active fraction: {referencemetadata['mass_flux_cap_fraction']:.0%}")

def makeparams(settings, updates=None):
    params = default_params(device=device)
    params.update(extract_param_overrides(load_run_config()))
    params.update({
        'dt': settings['dt'],
        'ps0': settings['surface_pressure'],
        'ts_init': settings['surface_temperature'],
        'solar_constant': settings['solar_constant'],
        'zenith_factor': settings['zenith_factor'],
        'ocean_depth': settings['ocean_depth'],
        'albedo': settings['surface_albedo'],
        'wind_speed': settings['wind_speed'],
        'convection_scheme': 'mass_flux',
        'radiation_scheme': 'multiband',
        'use_slab_ocean': True,
        'profile_diagnostics': True,
    })
    if updates is not None:
        params.update(updates)
    return params

def loadreference(nlevels=20, batch=1):
    reference = np.load(root / 'notebooks/data/atm407_equilibrium_20level.npz')
    settings = dict(experiment)
    settings['nlevels'] = nlevels
    grid = make_grid(nlevels, device=device)
    params = makeparams(settings)
    state = initial_state(batch, grid, params, device=device)
    sourcesigma = reference['sigma_full']
    targetsigma = grid['sigma_full'].cpu().numpy()

    for name in ['t', 'q', 'qc', 'cloud_fraction']:
        profile = np.interp(targetsigma, sourcesigma, reference[name])
        values = torch.as_tensor(profile, dtype=state[name].dtype, device=device)
        state[name] = values.unsqueeze(0).repeat(batch, 1)

    referencegrid = make_grid(len(sourcesigma), device=device)
    sourcedsigma = referencegrid['dsigma'].cpu().numpy()
    sourcewater = np.sum(reference['q'] * sourcedsigma)
    targetwater = torch.sum(state['q'][0] * grid['dsigma']).item()
    state['q'] = state['q'] * (sourcewater / targetwater)
    state['ts'].fill_(float(reference['ts']))
    state['ps'].fill_(float(reference['ps']))
    state['slab_ts_ref'] = state['ts'].clone()
    state['slab_energy'].zero_()
    return update_derived(state, grid)

def pressuregradient(field, pressure):
    gradient = torch.zeros_like(field)
    gradient[:, 1:-1] = (field[:, 2:] - field[:, :-2]) / (pressure[:, 2:] - pressure[:, :-2])
    gradient[:, 0] = (field[:, 1] - field[:, 0]) / (pressure[:, 1] - pressure[:, 0])
    gradient[:, -1] = (field[:, -1] - field[:, -2]) / (pressure[:, -1] - pressure[:, -2])
    return gradient

def ascentforcing(grid, peakomega, durationdays=1.0):
    peakomega = torch.as_tensor(peakomega, dtype=torch.float32, device=device).reshape(-1, 1)
    sigma = grid['sigma_full'].to(device=device, dtype=torch.float32).reshape(1, -1)
    shape = torch.sin(torch.pi * sigma).clamp(min=0.0)

    def forcing(step, state):
        if step * experiment['dt'] >= durationdays * 86400:
            return None
        pressure = state['p']
        omega = -peakomega.to(pressure.dtype) * shape.to(pressure.dtype)
        dtdp = pressuregradient(state['t'], pressure)
        dqdp = pressuregradient(state['q'], pressure)
        temperaturetendency = -omega * dtdp + (Rd / cp) * state['t'] * omega / pressure
        moisturetendency = -omega * dqdp
        return {'dt': temperaturetendency, 'dq': moisturetendency}

    return forcing

def integrate(settings, updates=None, batch=1, state=None, lsforcing=None):
    grid = make_grid(settings['nlevels'], device=device)
    params = makeparams(settings, updates)
    if state is None:
        state = initial_state(batch, grid, params, device=device)
    stepsperday = round(86400 / settings['dt'])
    nsteps = round(settings['days'] * stepsperday)
    diagnosticsteps = max(1, round(settings['diagnostic_hours'] * 3600 / settings['dt']))
    start = time.perf_counter()
    state, history = run(
        state, grid, params, nsteps,
        rad_interval=settings['radiation_steps'],
        diag_interval=diagnosticsteps,
        ls_forcing=lsforcing,
    )
    elapsed = time.perf_counter() - start
    return grid, params, state, history, elapsed

def series(history, name, member=0, scale=1.0):
    values = [entry[name][member].detach().cpu().item() for entry in history]
    return np.array(values) * scale

def days(history, dt):
    return np.array([entry['step'] for entry in history]) * dt / 86400

## Exercise 1: vertical coordinate and column mass

The model uses the terrain-following coordinate $\sigma=p/p_s$. The cell loads the near-equilibrium reference state. Plot full-level pressure and layer pressure thickness. Explain why $\Delta p/g$ is the mass per unit area of a hydrostatic layer. Then use the printed sum to verify that the discrete atmospheric mass is consistent with $p_s/g$.

In [ ]:
grid = make_grid(experiment['nlevels'], device=device)
params = makeparams(experiment)
state = loadreference(experiment['nlevels'])
pressure = state['p'][0].cpu().numpy() / 100
deltap = state['dp'][0].cpu().numpy() / 100
levels = np.arange(experiment['nlevels'])

fig, axes = plt.subplots(1, 2, figsize=(9, 5), sharey=True)
axes[0].plot(pressure, levels, marker='o')
axes[0].set_xlabel('full-level pressure (hPa)')
axes[1].barh(levels, deltap)
axes[1].set_xlabel('layer pressure thickness (hPa)')
axes[0].set_ylabel('model level')
axes[0].invert_yaxis()
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

massfromlayers = state['dp'].sum().item() / g
massfromsurface = state['ps'].item() / g
print(f'layer sum: {massfromlayers:.2f} kg m-2')
print(f'ps / g:    {massfromsurface:.2f} kg m-2')

## Exercise 2: diagnose the initial sounding

Compute potential temperature, geopotential height, relative humidity, and the dry Brunt--Vaisala frequency $N^2=(g/\theta)(\partial\theta/\partial z)$. Identify strongly stable, weakly stable, and any problematic layers. Why does potential temperature, rather than temperature, diagnose dry static stability? Quantify how much atmospheric mass is at or above 95% RH. Explain why a balanced idealized column should not be interpreted as an observed tropical mean sounding.


In [ ]:
temperature = state['t'][0]
pressurepa = state['p'][0]
theta = temperature * (100000.0 / pressurepa) ** (Rd / cp)
rh = relative_humidity(state['q'], state['t'], state['p'])[0] * 100
height = geopotential(state['t'], state['q'], state['p'], grid)[0]
dthetadz = np.gradient(theta.cpu().numpy(), height.cpu().numpy())
n2 = g / theta.cpu().numpy() * dthetadz

fig, axes = plt.subplots(1, 4, figsize=(14, 5), sharey=True)
axes[0].plot(temperature.cpu(), pressure)
axes[0].set_xlabel('temperature (K)')
axes[1].plot(theta.cpu(), pressure)
axes[1].set_xlabel('potential temperature (K)')
axes[2].plot(rh.cpu(), pressure)
axes[2].plot(rh[rh >= 95].cpu(), pressure[rh.cpu().numpy() >= 95], 'o', color='tab:red')
axes[2].axvline(95, color='tab:red', linestyle='--', linewidth=0.8)
axes[2].set_xlabel('relative humidity (%)')
axes[3].plot(n2 * 1e4, pressure)
axes[3].axvline(0, color='black', linewidth=0.8)
axes[3].set_xlabel(r'$N^2$ ($10^{-4}$ s$^{-2}$)')
axes[0].set_ylabel('pressure (hPa)')
axes[0].invert_yaxis()
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()
saturatedmass = state['dp'][0, rh >= 95].sum() / state['dp'][0].sum()
print(f'mass at or above 95% RH: {saturatedmass.item():.0%}')
print(f'height range: {height.min().item() / 1000:.1f} to {height.max().item() / 1000:.1f} km')
print(f'minimum dry N2: {n2.min():+.2e} s-2')
print('levels with dry N2 below zero (hPa):', np.round(pressure[n2 < 0], 1))

## Exercise 3: follow one physics timestep

A host dynamical core would call the column physics once per timestep. The SCM applies radiation, surface exchange, boundary-layer mixing, shallow convection, deep convection, condensation, and cloud microphysics sequentially. Radiation and surface exchange change the atmospheric column's total moist energy. The other schemes mainly redistribute temperature and moisture vertically, so their column-integrated moist-energy tendencies are designed to be nearly zero. The first panel shows the two boundary contributions; the profile panels reveal the internal tendencies that a column-integrated bar chart would hide.

In [ ]:
stepstate = deepcopy(state)
stepstate, diagnostics, radiationcache = physics_step(stepstate, grid, params)
boundarylabels = ['radiation', 'surface exchange']
boundaryvalues = [
    diagnostics['rad_energy_tendency'][0].item(),
    diagnostics['surface_energy_tendency'][0].item(),
]
processes = [
    ('radiation', 'radiation'),
    ('surface', 'surface'),
    ('boundary layer', 'boundary_layer'),
    ('shallow', 'shallow'),
    ('deep convection', 'deep'),
    ('condensation', 'condensation'),
    ('clouds', 'cloud'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['tab:red' if value > 0 else 'tab:blue' for value in boundaryvalues]
axes[0].bar(boundarylabels, boundaryvalues, color=colors)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_ylabel('atmospheric energy tendency (W m-2)')
axes[0].set_title('boundary contributions')
axes[0].tick_params(axis='x', rotation=20)
axes[0].grid(axis='y', alpha=0.3)

for label, name in processes:
    temperaturetendency = diagnostics[f'{name}_temperature_tendency'][0] * 86400
    moisturetendency = diagnostics[f'{name}_moisture_tendency'][0] * 86400 * 1000
    axes[1].plot(temperaturetendency.cpu(), pressure, label=label)
    axes[2].plot(moisturetendency.cpu(), pressure, label=label)
axes[1].set(xlabel='temperature tendency (K day-1)', ylabel='pressure (hPa)')
axes[2].set(xlabel='moisture tendency (g kg-1 day-1)', ylabel='pressure (hPa)')
for ax in axes[1:]:
    ax.axvline(0, color='black', linewidth=0.8)
    ax.invert_yaxis()
    ax.grid(alpha=0.3)
axes[2].legend(fontsize=8, loc='best')
fig.tight_layout()
plt.show()
print(f"TOA net flux: {diagnostics['toa_net'][0].item():+.2f} W m-2")
print(f"surface total flux: {diagnostics['surface_total_flux'][0].item():+.2f} W m-2")
print(f"column residual: {diagnostics['column_energy_residual'][0].item():+.2f} W m-2")

## Exercise 4: force the column with large-scale ascent

A dynamical core supplies vertical advection to the physics column. Prescribe an ascent profile with $\omega=Dp/Dt<0$, zero at the top and surface and strongest near $\sigma=0.5$. The imposed pressure-coordinate tendencies are $\partial T/\partial t=-\omega\,\partial T/\partial p+(R_d/c_p)T\omega/p$ and $\partial q/\partial t=-\omega\,\partial q/\partial p$. Apply the forcing for one day, then let the column adjust for two more days. Predict the signs of the temperature and moisture tendencies before running the cell.

In [ ]:
grid = make_grid(experiment['nlevels'], device=device)
peakomega = 0.05
forcing = ascentforcing(grid, [peakomega], durationdays=1.0)
previewstate = loadreference(experiment['nlevels'])
preview = forcing(0, previewstate)
omega = -peakomega * np.sin(np.pi * grid['sigma_full'].cpu().numpy())

fig, axes = plt.subplots(1, 3, figsize=(12, 5), sharey=True)
axes[0].plot(omega * 36, pressure)
axes[0].set(xlabel=r'$\omega$ (hPa hour$^{-1}$)', ylabel='pressure (hPa)')
axes[1].plot(preview['dt'][0].cpu() * 86400, pressure)
axes[1].set_xlabel('imposed temperature tendency (K day-1)')
axes[2].plot(preview['dq'][0].cpu() * 86400 * 1000, pressure)
axes[2].set_xlabel('imposed moisture tendency (g kg-1 day-1)')
for ax in axes:
    ax.axvline(0, color='black', linewidth=0.8)
    ax.invert_yaxis()
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

controlstate = loadreference(experiment['nlevels'])
grid, params, controlstate, controlhistory, controlelapsed = integrate(
    experiment, state=controlstate
)
forcedstate = loadreference(experiment['nlevels'])
grid, params, forcedstate, forcedhistory, forcedelapsed = integrate(
    experiment, state=forcedstate, lsforcing=forcing
)
timeaxis = days(controlhistory, experiment['dt'])

fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)
axes[0].plot(timeaxis, series(controlhistory, 'cape'), label='control')
axes[0].plot(timeaxis, series(forcedhistory, 'cape'), label='one day of ascent')
axes[0].set_ylabel('CAPE (J kg-1)')
axes[0].legend()
axes[1].plot(timeaxis, series(controlhistory, 'precip_conv', scale=86400), label='control deep rain')
axes[1].plot(timeaxis, series(forcedhistory, 'precip_conv', scale=86400), label='forced deep rain')
axes[1].set_ylabel('deep rain (mm day-1)')
axes[1].legend()
axes[2].plot(timeaxis, series(forcedhistory, 'forcing_energy_tendency'))
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set(xlabel='model day', ylabel='imposed energy tendency (W m-2)')
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()
print(f'maximum ascent: {-peakomega * 36:.2f} hPa hour-1')
print(f"control mean deep rain: {series(controlhistory, 'precip_conv', scale=86400).mean():.2f} mm day-1")
print(f"forced mean deep rain: {series(forcedhistory, 'precip_conv', scale=86400).mean():.2f} mm day-1")
print(f'total runtime: {controlelapsed + forcedelapsed:.1f} s')

## Exercise 5: challenge--dynamical forcing versus convective adjustment

Run nine columns spanning three ascent strengths and three convective CAPE-removal timescales. The ascent acts for one day and each column runs for two days. Before running the cell, predict where CAPE will accumulate and where convection will keep pace with the forcing. Interpret the result as a competition between a resolved dynamical forcing timescale and an unresolved convective-adjustment timescale.

All nine columns run simultaneously to keep the wait short. Check that the mass-flux cap remains inactive before interpreting the response.


In [ ]:
omegavalues = [0.02, 0.05, 0.10]
timescalevalues = [43200.0, 86400.0, 172800.0]
cases = [(omega, timescale) for omega in omegavalues for timescale in timescalevalues]
updates = {
    'tau_cape': torch.tensor([case[1] for case in cases], device=device),
}
challengesettings = dict(experiment)
challengesettings['days'] = 2
challengegrid = make_grid(experiment['nlevels'], device=device)
challengestart = loadreference(experiment['nlevels'], batch=len(cases))
challengeforcing = ascentforcing(
    challengegrid, [case[0] for case in cases], durationdays=1.0
)
grid, params, challengestate, challengehistory, elapsed = integrate(
    challengesettings, updates=updates, batch=len(cases), state=challengestart,
    lsforcing=challengeforcing,
)
capeanomalies = []
rainrates = []
capfractions = []
referencecape = referencemetadata['cape_jkg']
for member, case in enumerate(cases):
    cape = series(challengehistory, 'cape', member=member)
    capeanomalies.append(cape.max() - referencecape)
    rainrates.append(series(challengehistory, 'precip_conv', member=member, scale=86400).mean())
    capfractions.append(series(challengehistory, 'mass_flux_cap_active', member=member).mean())

capegrid = np.array(capeanomalies).reshape(len(omegavalues), len(timescalevalues))
raingrid = np.array(rainrates).reshape(len(omegavalues), len(timescalevalues))
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax, values, title, label, colormap in [
    (axes[0], capegrid, 'maximum CAPE anomaly', 'CAPE anomaly (J kg-1)', 'coolwarm'),
    (axes[1], raingrid, 'mean deep precipitation', 'rain (mm day-1)', 'viridis'),
]:
    image = ax.imshow(values, origin='lower', aspect='auto', cmap=colormap)
    ax.set_xticks(range(len(timescalevalues)), [f'{value / 3600:.0f}' for value in timescalevalues])
    ax.set_yticks(range(len(omegavalues)), [f'{value * 36:.2f}' for value in omegavalues])
    ax.set(xlabel='convective timescale (hours)', ylabel='peak ascent magnitude (hPa hour-1)', title=title)
    fig.colorbar(image, ax=ax, label=label)
plt.show()

print(f'maximum cap-active fraction: {max(capfractions):.0%}')
winner = int(np.argmax(capeanomalies))
print('largest CAPE accumulation (omega, timescale):', cases[winner])
print(f'maximum CAPE anomaly: {capeanomalies[winner]:.1f} J kg-1')
print(f'batched challenge runtime: {elapsed:.1f} s')

## Optional extension: numerical sensitivity

Interpolate the 20-level reference state to 10, 20, and 40 levels, then give each grid one day to adjust. Compare CAPE before and after adjustment, precipitation, and runtime. Surface coupling, boundary-layer depth, subcloud export, plume entrainment, and plume detrainment are expressed in resolution-aware pressure or sigma coordinates, but an interpolated state is still not a native-grid equilibrium. This is therefore a remapping stress test rather than a formal convergence test. Explain why CAPE can still change when the same sounding is sampled on another grid, and describe the separate native-grid integrations that would be required for a convergence claim.

In [ ]:
resolutionresults = []
for nlevels in [10, 20, 40]:
    settings = dict(experiment)
    settings['nlevels'] = nlevels
    settings['days'] = 1
    initialstate = loadreference(nlevels)
    diagnosticstate = deepcopy(initialstate)
    diagnosticgrid = make_grid(nlevels, device=device)
    diagnosticparams = makeparams(settings)
    diagnosticstate, initialdiagnostics, cache = physics_step(
        diagnosticstate, diagnosticgrid, diagnosticparams
    )
    grid, params, finalstate, history, elapsed = integrate(
        settings, state=initialstate
    )
    resolutionresults.append({
        'levels': nlevels,
        'initialcape': initialdiagnostics['cape'][0].item(),
        'cape': series(history, 'cape')[-1],
        'rain': series(history, 'precip_total', scale=86400).mean(),
        'runtime': elapsed,
    })

for result in resolutionresults:
    print(
        f"{result['levels']:2d} levels | "
        f"CAPE {result['initialcape']:7.1f} -> {result['cape']:7.1f} J kg-1 | "
        f"mean rain {result['rain']:5.2f} mm day-1 | "
        f"runtime {result['runtime']:4.1f} s"
    )

## Submission

Submit your completed notebook with: (1) your hydrostatic-mass and static-stability analysis, (2) an interpretation of the physics-tendency profiles, (3) your prediction and explanation for the ascent/convective-timescale challenge, and (4) one paragraph explaining what a single-column model cannot represent without a dynamical core. The numerical-sensitivity extension is optional unless assigned by your instructor.